# COMP34812 NLI — Demo Notebook

**Group:** 16  
**Category:** B — Deep Learning (non-transformer)  
**Model:** ResESIM + 1477d input embeddings

| Component | Dim | Reference |
|---|---|---|
| GloVe | 300 | Pennington et al., 2014 |
| ELMo | 1024 | Peters et al., 2018 |
| CharCNN | 100 | Kim et al., 2016 |
| POS embeddings | 50 | Universal Dependencies |
| Negation flags | 3 | spaCy dep parse + boundary |
| **Total** | **1477** | |

## Instructions

1. Run all cells top to bottom
2. Predictions saved to `Group_16_B.csv`

**Input CSV:** must have `premise` and `hypothesis` columns

```
test.csv
    ↓  ELMo pre-computation
    ↓  GloVe + CharCNN + POS + Negation
    ↓  inference_embeddings.npz  [N, 64, 1477]
    ↓  OracleNet (ResESIM)
    ↓  Group_16_B.csv
```

## 1. Environment Setup

**Skip this cell if running locally** — only needed on Colab.

In [2]:
# ── Create Python 3.10 venv for AllenNLP ELMo ────────────────────────────────
import os, sys

if not os.path.exists('./venv310/bin/python'):
    !{sys.executable} -m pip install -q virtualenv
    !virtualenv -p python3.10 ./venv310 -q

# Always ensure required packages are installed in venv310
!./venv310/bin/pip install -q \
    'tokenizers==0.13.3' \
    'allennlp==2.10.1' \
    'allennlp-models==2.10.1' \
    'numpy<2.0'
print('venv310 ready.')

# ── Install packages into current kernel ─────────────────────────────────────
!{sys.executable} -m pip install -q spacy scikit-learn
!{sys.executable} -m spacy download en_core_web_sm -q
print('Kernel packages ready.')

venv310 ready.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
/Users/kaanoktem/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
Kernel packages ready.


## 2. Setup Path + Verify Files

In [3]:
import sys
sys.path.append('.')

# Install required packages
!{sys.executable} -m pip install -q spacy scikit-learn
!{sys.executable} -m spacy download en_core_web_sm -q

print('Ready.')

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
/Users/kaanoktem/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
Ready.


In [4]:
# ── Verify required files ─────────────────────────────────────────────────────
import os

required = {
    './notebook_data/meta.pt': 'Vocab + GloVe meta',
    './final_model_versions/ff2f02d4/best_model.pt': 'Trained model weights',
    './data/test.csv': 'Test data for inference',
    './notebook_data/elmo_model/options.json': 'ELMo options',
    './notebook_data/elmo_model/weights.hdf5': 'ELMo weights',
}

all_ok = True
for path, desc in required.items():
    exists = os.path.exists(path)
    size   = f'  ({os.path.getsize(path)/1e6:.1f} MB)' if exists else ''
    status = '✓' if exists else '✗  MISSING'
    print(f'  {status}  {path}{size}')
    if not exists: all_ok = False

print()
print('All files present — continue.' if all_ok else 'Fix missing files before continuing.')

  ✓  ./notebook_data/meta.pt  (1.0 MB)
  ✓  ./final_model_versions/ff2f02d4/best_model.pt  (86.9 MB)
  ✓  ./data/test.csv  (0.4 MB)
  ✓  ./notebook_data/elmo_model/options.json  (0.0 MB)
  ✓  ./notebook_data/elmo_model/weights.hdf5  (374.4 MB)

All files present — continue.


## 3. Pre-compute Embeddings

Uses `EmbeddingPrecomputer` from `precomputeClasses.py`.  
Steps run automatically:
1. Load meta (vocab, char2idx, pos2idx, GloVe)
2. Build embedding layers (CharCNN, POS, GloVe)
3. ELMo pre-computation via Python 3.10 venv
4. Full 1477d pre-computation
5. Save `inference_embeddings.npz`

In [5]:
from precomputeClasses import EmbeddingPrecomputer

pc = EmbeddingPrecomputer(
    meta_path    = './notebook_data/meta.pt',
    elmo_options = './notebook_data/elmo_model/options.json',
    elmo_weights = './notebook_data/elmo_model/weights.hdf5',
    elmo_venv    = './venv310/bin/python',
)

pc.run(
    csv_path   = './data/test.csv',
    output_npz = './notebook_data/inference_embeddings.npz',
)


  EmbeddingPrecomputer — COMP34812 NLI
  Input:  ./data/test.csv
  Output: ./notebook_data/inference_embeddings.npz
  Dim:    1477  (GloVe+ELMo+CharCNN+POS+Neg)

[1/5] Loading meta from notebook_data/meta.pt...
      vocab=566  char=256  pos=18
[2/5] Building embedding layers...


/Users/kaanoktem/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


      GloVe frozen  |  CharCNN  |  POS  |  spaCy en_core_web_sm
[3/5] Pre-computing ELMo (Peters et al., 2018)...


/Users/kaanoktem/NLU-CW-test-demo/venv310/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


ELMo on cpu
  3302 premises...
    64/3302
    704/3302
    1344/3302
    1984/3302
    2624/3302
    3264/3302
  3302 hypotheses...
    64/3302
    704/3302
    1344/3302
    1984/3302
    2624/3302
    3264/3302
ELMo done.
      ELMo saved.
[4/5] Pre-computing 1477d embeddings...


  batches: 100%|██████████| 13/13 [01:16<00:00,  5.86s/it]


[5/5] Saving ./notebook_data/inference_embeddings.npz...
      Saved: 0.35 GB  |  shape (3302, 64, 1477)
      NaN check: premise=0 ✓  hypothesis=0 ✓

✓ Pre-computation complete — ./notebook_data/inference_embeddings.npz


'./notebook_data/inference_embeddings.npz'

## 4. Load Model + Run Inference

In [6]:
# ── Select device ─────────────────────────────────────────────────────────────
import torch

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Device: {device}')

Device: mps


In [7]:
# ── Load dataset and dataloader ────────────────────────────────────────────────
from res_esim.loader.res_esim_dataset import ResESIM_Dataset
from torch.utils.data import DataLoader

dataset = ResESIM_Dataset('./notebook_data/inference_embeddings.npz')
loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

Loading ./notebook_data/inference_embeddings.npz...
  3302 samples, dim=1477


In [8]:
# ── Load OracleNet model ──────────────────────────────────────────────────────
from res_esim.model_layers.oracle_net import OracleNet

model = OracleNet(
    input_dim=1477,
    hidden_dim=512,
    num_blocks=3,
    num_classes=2,
    dropout_rate=0.2,
    num_heads=8,
)
model.load_state_dict(torch.load(
    './final_model_versions/ff2f02d4/best_model.pt',
    map_location=device,
    weights_only=False,
))
model.to(device).eval()
print('Model loaded and set to eval mode.')

Model loaded and set to eval mode.


In [9]:
# ── Run inference ─────────────────────────────────────────────────────────────
from tqdm import tqdm

all_preds = []
with torch.no_grad():
    for batch in tqdm(loader, desc='inference'):
        logits = model(
            batch['premise_embedding'].to(device),
            batch['hyp_embedding'].to(device),
            batch['premise_length'].to(device),
            batch['hyp_length'].to(device),
        )
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())

print(f'Inference complete: {len(all_preds)} predictions')

inference: 100%|██████████| 104/104 [00:16<00:00,  6.22it/s]

Inference complete: 3302 predictions


In [10]:
# ── Save predictions ──────────────────────────────────────────────────────────
import pandas as pd

pd.DataFrame({'prediction': all_preds}).to_csv('Group_16_B.csv', index=False)

print(f'Saved Group_16_B.csv  ({len(all_preds)} predictions)')
print(f'Label distribution: {pd.Series(all_preds).value_counts().sort_index().to_dict()}')

Saved Group_16_B.csv  (3302 predictions)
Label distribution: {0: 2071, 1: 1231}


## Done

Predictions saved to `Group_n_B.csv`  
Download and submit via Canvas.